In [4]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path.cwd().parent                     # il notebook sta in results/
ENC_DIR = PROJECT_ROOT / "data" / "encoded"

con = duckdb.connect()
con.execute("SET threads TO 8")
con.execute("SET memory_limit = '8GB'")

for nome in ["train_enc", "tune_enc", "val_enc"]:
    file = (ENC_DIR / f"{nome}.parquet").as_posix()
    con.execute(f"CREATE OR REPLACE VIEW {nome} AS SELECT * FROM read_parquet('{file}')")

display(con.execute("""
    SELECT 'train_enc' AS insieme, COUNT(*) AS n FROM train_enc
    UNION ALL SELECT 'tune_enc', COUNT(*) FROM tune_enc
    UNION ALL SELECT 'val_enc',  COUNT(*) FROM val_enc
""").df())

display(con.execute("DESCRIBE val_enc").df()[["column_name", "column_type"]])

,insieme,n
0,train_enc,3000316
1,tune_enc,1000351
2,val_enc,10069254


,column_name,column_type
0,riga,BIGINT
1,flow_id,VARCHAR
2,t_start,TIMESTAMP WITH TIME ZONE
3,delta_rtt,DOUBLE
4,start_rtt,DOUBLE
5,id_ip_version,INTEGER
6,id_experiment,INTEGER
7,id_exp_act,INTEGER
8,id_application,INTEGER
9,id_hostname,INTEGER


Leggo da meta.json l’elenco dei 13 input categorici (id_*) e carico, per ogni insieme, solo target (delta_rtt), start_rtt e codici, come array numpy. Le colonne sono elencate esplicitamente: an_durata_s (durata, nota solo a fine flow) non deve mai entrare negli input. Per la validation tengo anche riga, la chiave per riattaccare le previsioni ai flow.

In [5]:
import json
import numpy as np

meta = json.loads((ENC_DIR / "meta.json").read_text(encoding="utf-8"))
COL_ID = meta["colonne_id"]
print(len(COL_ID), "input categorici:", COL_ID)

def carica(nome):
    df = con.execute(f"SELECT riga, delta_rtt, start_rtt, {', '.join(COL_ID)} FROM {nome}").df()
    riga = df["riga"].to_numpy()
    y    = df["delta_rtt"].to_numpy(np.float32)
    rtt  = df["start_rtt"].to_numpy(np.float32)
    X    = df[COL_ID].to_numpy(np.int32)
    return riga, y, rtt, X

_,       y_tr, rtt_tr, X_tr = carica("train_enc")
_,       y_tu, rtt_tu, X_tu = carica("tune_enc")
riga_va, y_va, rtt_va, X_va = carica("val_enc")

print("forme di X:", X_tr.shape, X_tu.shape, X_va.shape)
print(f"memoria X_va: {X_va.nbytes / 1e9:.2f} GB")
print("mediana di y_tr:", np.median(y_tr))

13 input categorici: ['id_ip_version', 'id_experiment', 'id_exp_act', 'id_application', 'id_hostname', 'id_server_site', 'id_client_site', 'id_server_L2', 'id_client_L2', 'id_porta_servizio', 'id_porta_client', 'id_verso', 'id_verso_incerto']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

forme di X: (3000316, 13) (1000351, 13) (10069254, 13)
memoria X_va: 0.52 GB
mediana di y_tr: 4.405


2. Trasformazione di start_rtt

Nel modello lineare start_rtt entra con un solo coefficiente. Uso log(start_rtt) perché conti il rapporto tra RTT e non la differenza in ms, e per comprimere la coda lunga. Poi standardizzo con media μ e deviazione σ calcolate solo sul train, applicate identiche a tune e validation. Prima controllo che non ci siano RTT nulli (il log di 0 non esiste).

In [6]:
print("start_rtt minimo:", rtt_tr.min(), rtt_tu.min(), rtt_va.min())
assert min(rtt_tr.min(), rtt_tu.min(), rtt_va.min()) > 0, "start_rtt nullo: il log non è definito"

log_tr = np.log(rtt_tr)
mu, sigma = float(log_tr.mean()), float(log_tr.std())

def trasforma_rtt(rtt):
    return ((np.log(rtt) - mu) / sigma).astype(np.float32)

x_tr = trasforma_rtt(rtt_tr)
x_tu = trasforma_rtt(rtt_tu)
x_va = trasforma_rtt(rtt_va)

print(f"mu = {mu:.3f} (RTT tipico ≈ {np.exp(mu):.1f} ms) | sigma = {sigma:.3f}")
for nome, x in [("train", x_tr), ("tune", x_tu), ("val", x_va)]:
    print(f"{nome:5s} media = {x.mean():+.3f} | std = {x.std():.3f}")

start_rtt minimo: 0.007 0.007 0.008
mu = 2.823 (RTT tipico ≈ 16.8 ms) | sigma = 1.653
train media = +0.000 | std = 1.000
tune  media = -0.094 | std = 0.970
val   media = -0.010 | std = 0.907


3. Il modello

$\hat y = b + w\, x_{\text{rtt}} + \sum_j \theta_j[\text{codice}_j]$: intercetta, coefficiente di start_rtt e un coefficiente per ogni valore di ogni campo (una tabella nn.Embedding(n, 1) per colonna, separata anche per server e client). Le dimensioni delle tabelle vengono da meta.json (4 codici riservati + valori frequenti). I coefficienti partono da 0 e l’intercetta dalla mediana del train: il modello non addestrato coincide con B0, e lo verifico.

In [8]:
import torch
import torch.nn as nn

vocab = json.loads((ENC_DIR / "vocabolari.json").read_text(encoding="utf-8"))

# colonna -> vocabolario (es. 'server_site' -> 'site') -> numero di codici
col_vocab = {c: nome for nome, colonne in vocab["colonne"].items() for c in colonne}
N_CODICI  = [meta["n_codici"][col_vocab[c.removeprefix("id_")]] for c in COL_ID]

class Lineare(nn.Module):
    def __init__(self, n_codici, intercetta_iniziale):
        super().__init__()
        self.emb = nn.ModuleList([nn.Embedding(n, 1) for n in n_codici])
        for e in self.emb:
            nn.init.zeros_(e.weight)
        self.w = nn.Parameter(torch.zeros(1))
        self.b = nn.Parameter(torch.tensor([intercetta_iniziale]))

    def forward(self, x_cat, x_rtt):
        out = self.b + self.w * x_rtt
        for j, e in enumerate(self.emb):
            out = out + e(x_cat[:, j]).squeeze(1)
        return out

MEDIANA_TR = float(np.median(y_tr))
modello = Lineare(N_CODICI, MEDIANA_TR)

print(dict(zip(COL_ID, N_CODICI)))
print("parametri:", sum(p.numel() for p in modello.parameters()))

# controllo: il modello non addestrato deve prevedere la mediana per ogni flow
with torch.no_grad():
    prova = modello(torch.from_numpy(X_tr[:5]).long(), torch.from_numpy(x_tr[:5]))
print("previsioni iniziali:", prova.numpy(), "| mediana:", MEDIANA_TR)

{'id_ip_version': 6, 'id_experiment': 10, 'id_exp_act': 33, 'id_application': 60, 'id_hostname': 690, 'id_server_site': 87, 'id_client_site': 87, 'id_server_L2': 344, 'id_client_L2': 344, 'id_porta_servizio': 50, 'id_porta_client': 8, 'id_verso': 6, 'id_verso_incerto': 6}
parametri: 1733
previsioni iniziali: [4.405 4.405 4.405 4.405 4.405] | mediana: 4.40500020980835


4. Addestramento

prevedi calcola le previsioni a blocchi (senza gradienti). addestra(lam) crea un modello nuovo e lo addestra con Adam su mini-batch di train_enc in ordine casuale, con loss L1 (MAE). La penalità L2 di intensità lam è applicata solo ai coefficienti delle tabelle, non a intercetta e w. A fine epoca misura la MAE su tune_enc, salva i coefficienti migliori e si ferma dopo pazienza epoche senza miglioramenti (early stopping). Prima prova: lam = 0, per verificare che l’addestramento funzioni

In [10]:
import copy
import time

# train in tensori PyTorch una volta sola (gli indici di nn.Embedding devono essere int64)
Xt_tr = torch.from_numpy(X_tr).long()
xt_tr = torch.from_numpy(x_tr)
yt_tr = torch.from_numpy(y_tr)

@torch.no_grad()
def prevedi(modello, X, x, blocco=500_000):
    """Previsioni a blocchi, per non creare tensori enormi (la validation ha 10 M righe)."""
    modello.eval()
    out = [modello(torch.from_numpy(X[i:i + blocco]).long(), torch.from_numpy(x[i:i + blocco]))
           for i in range(0, len(X), blocco)]
    return torch.cat(out).numpy()

def addestra(lam, lr=0.01, batch=4096, max_epoche=30, pazienza=3, seme=0):
    torch.manual_seed(seme)
    modello = Lineare(N_CODICI, MEDIANA_TR)
    opt = torch.optim.Adam([
        {"params": modello.emb.parameters(),   "weight_decay": lam},   # coefficienti: penalizzati
        {"params": [modello.w, modello.b],     "weight_decay": 0.0},   # w e intercetta: no
    ], lr=lr)

    n = len(yt_tr)
    mae_migliore, epoca_migliore, stato_migliore, storia = float("inf"), 0, None, []

    for epoca in range(1, max_epoche + 1):
        t0 = time.time()
        modello.train()
        ordine = torch.randperm(n)
        for i in range(0, n, batch):
            idx  = ordine[i:i + batch]
            pred = modello(Xt_tr[idx], xt_tr[idx])
            loss = (pred - yt_tr[idx]).abs().mean()
            opt.zero_grad()
            loss.backward()
            opt.step()

        mae_tu = float(np.abs(prevedi(modello, X_tu, x_tu) - y_tu).mean())
        storia.append({"epoca": epoca, "mae_tune": mae_tu, "secondi": time.time() - t0})
        print(f"epoca {epoca:2d} | MAE tune = {mae_tu:.4f} | {time.time() - t0:.0f} s")

        if mae_tu < mae_migliore - 1e-4:
            mae_migliore, epoca_migliore = mae_tu, epoca
            stato_migliore = copy.deepcopy(modello.state_dict())
        elif epoca - epoca_migliore >= pazienza:
            print(f"early stopping: migliore all'epoca {epoca_migliore}")
            break

    modello.load_state_dict(stato_migliore)
    return modello, epoca_migliore, mae_migliore, pd.DataFrame(storia)

import pandas as pd
mae_b0_tune = float(np.abs(MEDIANA_TR - y_tu).mean())
print(f"riferimento B0 sul tune: MAE = {mae_b0_tune:.4f}")

modello, ep, mae_tu, storia = addestra(lam=0.0)

riferimento B0 sul tune: MAE = 3.2094
epoca  1 | MAE tune = 1.7681 | 9 s
epoca  2 | MAE tune = 1.7639 | 9 s
epoca  3 | MAE tune = 1.7674 | 9 s
epoca  4 | MAE tune = 1.7653 | 9 s
epoca  5 | MAE tune = 1.7676 | 9 s
early stopping: migliore all'epoca 2


4. Addestramento

prevedi calcola le previsioni a blocchi (senza gradienti). addestra(lam) crea un modello nuovo e lo addestra con Adam su mini-batch di train_enc in ordine casuale, con loss L1 (MAE). La penalità L2 di intensità lam è applicata solo ai coefficienti delle tabelle, non a intercetta e w. A fine epoca misura la MAE su tune_enc, salva i coefficienti migliori e si ferma dopo pazienza epoche senza miglioramenti (early stopping). Opzione gamma: a fine epoca il passo lr viene moltiplicato per gamma (es. 0,5 = dimezzato), così le ultime epoche fanno correzioni sempre più piccole e i coefficienti si assestano invece di oscillare intorno all’ottimo.

In [12]:
import copy
import time
import pandas as pd

# train in tensori PyTorch una volta sola (gli indici di nn.Embedding devono essere int64)
Xt_tr = torch.from_numpy(X_tr).long()
xt_tr = torch.from_numpy(x_tr)
yt_tr = torch.from_numpy(y_tr)

@torch.no_grad()
def prevedi(modello, X, x, blocco=500_000):
    """Previsioni a blocchi, per non creare tensori enormi (la validation ha 10 M righe)."""
    modello.eval()
    out = [modello(torch.from_numpy(X[i:i + blocco]).long(), torch.from_numpy(x[i:i + blocco]))
           for i in range(0, len(X), blocco)]
    return torch.cat(out).numpy()

def addestra(lam, lr=0.01, gamma=1.0, batch=4096, max_epoche=30, pazienza=3, seme=0):
    torch.manual_seed(seme)
    modello = Lineare(N_CODICI, MEDIANA_TR)
    opt = torch.optim.Adam([
        {"params": modello.emb.parameters(),   "weight_decay": lam},   # coefficienti: penalizzati
        {"params": [modello.w, modello.b],     "weight_decay": 0.0},   # w e intercetta: no
    ], lr=lr)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=gamma)   # lr × gamma a ogni epoca

    n = len(yt_tr)
    mae_migliore, epoca_migliore, stato_migliore, storia = float("inf"), 0, None, []

    for epoca in range(1, max_epoche + 1):
        t0 = time.time()
        modello.train()
        ordine = torch.randperm(n)
        for i in range(0, n, batch):
            idx  = ordine[i:i + batch]
            pred = modello(Xt_tr[idx], xt_tr[idx])
            loss = (pred - yt_tr[idx]).abs().mean()
            opt.zero_grad()
            loss.backward()
            opt.step()

        mae_tu = float(np.abs(prevedi(modello, X_tu, x_tu) - y_tu).mean())
        lr_epoca = scheduler.get_last_lr()[0]
        storia.append({"epoca": epoca, "lr": lr_epoca, "mae_tune": mae_tu, "secondi": time.time() - t0})
        print(f"epoca {epoca:2d} | lr = {lr_epoca:.5f} | MAE tune = {mae_tu:.4f} | {time.time() - t0:.0f} s")
        scheduler.step()

        if mae_tu < mae_migliore - 1e-4:
            mae_migliore, epoca_migliore = mae_tu, epoca
            stato_migliore = copy.deepcopy(modello.state_dict())
        elif epoca - epoca_migliore >= pazienza:
            print(f"early stopping: migliore all'epoca {epoca_migliore}")
            break

    modello.load_state_dict(stato_migliore)
    return modello, epoca_migliore, mae_migliore, pd.DataFrame(storia)

mae_b0_tune = float(np.abs(MEDIANA_TR - y_tu).mean())
print(f"riferimento B0 sul tune: MAE = {mae_b0_tune:.4f}")

riferimento B0 sul tune: MAE = 3.2094


4.1 Passo decrescente

Stesso addestramento (λ = 0), con il passo dimezzato a ogni epoca. Confronto con la prima prova (passo fisso, migliore MAE tune 1,7639): ci si aspetta una MAE che scende e si stabilizza invece di oscillare.

In [13]:
modello, ep, mae_tu, storia = addestra(lam=0.0, gamma=0.5, max_epoche=10)
print(f"migliore: epoca {ep}, MAE tune = {mae_tu:.4f}  (passo fisso: 1.7639)")

epoca  1 | lr = 0.01000 | MAE tune = 1.7681 | 9 s
epoca  2 | lr = 0.00500 | MAE tune = 1.7637 | 9 s
epoca  3 | lr = 0.00250 | MAE tune = 1.7640 | 9 s
epoca  4 | lr = 0.00125 | MAE tune = 1.7639 | 9 s
epoca  5 | lr = 0.00063 | MAE tune = 1.7632 | 9 s
epoca  6 | lr = 0.00031 | MAE tune = 1.7632 | 9 s
epoca  7 | lr = 0.00016 | MAE tune = 1.7631 | 9 s
epoca  8 | lr = 0.00008 | MAE tune = 1.7631 | 9 s
epoca  9 | lr = 0.00004 | MAE tune = 1.7632 | 9 s
epoca 10 | lr = 0.00002 | MAE tune = 1.7631 | 9 s
early stopping: migliore all'epoca 7
migliore: epoca 7, MAE tune = 1.7631  (passo fisso: 1.7639)


5. Scelta della penalità λ

Addestro il modello (passo dimezzato a ogni epoca, al massimo 8 epoche) con λ ∈ {0, 10⁻⁶, 10⁻⁵, 10⁻⁴, 10⁻³} e confronto le MAE sul tune. La penalità spinge i coefficienti verso 0 con la stessa forza per tutti i valori, mentre i dati li spingono con forza proporzionale a quanti flow li usano: agisce quindi soprattutto sui valori rari. Regola fissata in anticipo: vince la MAE più bassa; in caso di valori entro 0,0005 dal migliore, scelgo il λ più grande (più prudente).

In [14]:
LAMBDA = [0.0, 1e-6, 1e-5, 1e-4, 1e-3]
risultati_lam, modelli_lam = [], {}

for lam in LAMBDA:
    print(f"\n--- lam = {lam:g} ---")
    m, ep, mae, _ = addestra(lam=lam, gamma=0.5, max_epoche=8)
    modelli_lam[lam] = m
    risultati_lam.append({"lam": lam, "epoca_migliore": ep, "mae_tune": mae})

tab_lam = pd.DataFrame(risultati_lam)
tab_lam["distanza_dal_migliore"] = tab_lam["mae_tune"] - tab_lam["mae_tune"].min()

candidati = tab_lam[tab_lam["distanza_dal_migliore"] <= 0.0005]
LAM_SCELTO = float(candidati["lam"].max())

display(tab_lam.round(5))
print("lambda scelto:", LAM_SCELTO)


--- lam = 0 ---
epoca  1 | lr = 0.01000 | MAE tune = 1.7681 | 9 s
epoca  2 | lr = 0.00500 | MAE tune = 1.7637 | 9 s
epoca  3 | lr = 0.00250 | MAE tune = 1.7640 | 9 s
epoca  4 | lr = 0.00125 | MAE tune = 1.7639 | 9 s
epoca  5 | lr = 0.00063 | MAE tune = 1.7632 | 9 s
epoca  6 | lr = 0.00031 | MAE tune = 1.7632 | 9 s
epoca  7 | lr = 0.00016 | MAE tune = 1.7631 | 9 s
epoca  8 | lr = 0.00008 | MAE tune = 1.7631 | 9 s

--- lam = 1e-06 ---
epoca  1 | lr = 0.01000 | MAE tune = 1.7684 | 9 s
epoca  2 | lr = 0.00500 | MAE tune = 1.7637 | 9 s
epoca  3 | lr = 0.00250 | MAE tune = 1.7642 | 9 s
epoca  4 | lr = 0.00125 | MAE tune = 1.7639 | 9 s
epoca  5 | lr = 0.00063 | MAE tune = 1.7631 | 9 s
epoca  6 | lr = 0.00031 | MAE tune = 1.7632 | 9 s
epoca  7 | lr = 0.00016 | MAE tune = 1.7631 | 9 s
epoca  8 | lr = 0.00008 | MAE tune = 1.7631 | 9 s
early stopping: migliore all'epoca 5

--- lam = 1e-05 ---
epoca  1 | lr = 0.01000 | MAE tune = 1.7672 | 9 s
epoca  2 | lr = 0.00500 | MAE tune = 1.7634 | 9 s
epoc

,lam,epoca_migliore,mae_tune,distanza_dal_migliore
0,0.00000,7,1.76309,0.00022
1,0.00000,5,1.76314,0.00028
2,0.00001,5,1.76286,0.00000
3,0.00010,5,1.76567,0.00280
4,0.00100,5,1.78217,0.01930


lambda scelto: 1e-05


6. Modello finale: riaddestramento su train + tune

Con le impostazioni scelte sul tune (λ = LAM_SCELTO, passo dimezzato a ogni epoca, numero di epoche = epoca migliore per quel λ), riaddestro una sola volta su train_enc + tune_enc. Così il modello impara anche dalle 12 ore subito prima della validation. Senza early stopping: il tune ora è tra i dati di addestramento e non può più fare da giudice. Come controllo stampo la MAE media sui mini-batch di ogni epoca. μ e σ di start_rtt restano quelli di train_enc.


In [15]:
EPOCHE = int(tab_lam.loc[tab_lam["lam"] == LAM_SCELTO, "epoca_migliore"].iloc[0])
print(f"impostazioni finali: lambda = {LAM_SCELTO:g}, epoche = {EPOCHE}, gamma = 0.5")

# train + tune, uno sotto l'altro
Xt_tt = torch.cat([Xt_tr, torch.from_numpy(X_tu).long()])
xt_tt = torch.cat([xt_tr, torch.from_numpy(x_tu)])
yt_tt = torch.cat([yt_tr, torch.from_numpy(y_tu)])
print("flow di addestramento:", len(yt_tt))

def addestra_finale(lam, n_epoche, lr=0.01, gamma=0.5, batch=4096, seme=0):
    torch.manual_seed(seme)
    modello = Lineare(N_CODICI, float(yt_tt.median()))
    opt = torch.optim.Adam([
        {"params": modello.emb.parameters(),   "weight_decay": lam},
        {"params": [modello.w, modello.b],     "weight_decay": 0.0},
    ], lr=lr)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=gamma)

    n = len(yt_tt)
    for epoca in range(1, n_epoche + 1):
        t0 = time.time()
        modello.train()
        ordine = torch.randperm(n)
        somma_errori = 0.0
        for i in range(0, n, batch):
            idx  = ordine[i:i + batch]
            pred = modello(Xt_tt[idx], xt_tt[idx])
            loss = (pred - yt_tt[idx]).abs().mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            somma_errori += loss.item() * len(idx)

        print(f"epoca {epoca} | lr = {scheduler.get_last_lr()[0]:.5f} | "
              f"MAE media sui batch = {somma_errori / n:.4f} | {time.time() - t0:.0f} s")
        scheduler.step()
    return modello

t0 = time.time()
modello_finale = addestra_finale(LAM_SCELTO, EPOCHE)
TEMPO_TRAIN_B2 = time.time() - t0
print(f"tempo di addestramento: {TEMPO_TRAIN_B2:.0f} s")

impostazioni finali: lambda = 1e-05, epoche = 5, gamma = 0.5
flow di addestramento: 4000667
epoca 1 | lr = 0.01000 | MAE media sui batch = 1.9964 | 10 s
epoca 2 | lr = 0.00500 | MAE media sui batch = 1.9259 | 14 s
epoca 3 | lr = 0.00250 | MAE media sui batch = 1.9247 | 14 s
epoca 4 | lr = 0.00125 | MAE media sui batch = 1.9242 | 16 s
epoca 5 | lr = 0.00063 | MAE media sui batch = 1.9240 | 16 s
tempo di addestramento: 72 s
